In [1]:
import gzip, json, os, pandas as pd
from tqdm import tqdm

NOM_FICHIER = "/home/dina/Documents/MEMOIRE BIBLIO/Jstor/jstor_metadata_2026-05-03.jsonl.gz"
CSV_POETES = "/home/dina/Documents/MEMOIRE BIBLIO/poetes_LA.csv"
OUTPUT = "jstor_auteurs_revues.json"

REVUES = ["poetry", "the american poetry review", "the kenyon review", "prairie schooner"]
ANNEE_MIN = 1948

# Charger les 625 poètes
df = pd.read_csv(CSV_POETES, encoding="utf-8-sig")
poetes = set(df["nom_canonique"].str.lower().str.strip())
print(f"{len(poetes)} poètes chargés")

# Lire le fichier JSTOR
taille = os.path.getsize(NOM_FICHIER)
log = {r: {} for r in REVUES}

with gzip.open(NOM_FICHIER, "rb") as f:
    with tqdm(total=taille, unit="o", unit_scale=True, desc="Lecture") as barre:
        pos_prev = 0
        for ligne in f:
            pos = f.fileobj.tell()
            barre.update(pos - pos_prev)
            pos_prev = pos

            try:
                data = json.loads(ligne.decode("utf-8"))
            except:
                continue

            if data.get("content_type") != "article":
                continue

            is_part_of = str(data.get("is_part_of") or "").lower().strip()
            if is_part_of not in REVUES:
                continue

            annee_raw = str(data.get("published_date") or "")[:4]
            try:
                annee = int(annee_raw)
            except:
                continue
            if annee < ANNEE_MIN:
                continue

            # Séparer les auteurs multiples
            creators_string = str(data.get("creators_string") or "")
            auteurs = [a.strip() for a in creators_string.split(",") if a.strip()]

            for auteur in auteurs:
                if auteur.lower() in poetes:
                    if auteur not in log[is_part_of]:
                        log[is_part_of][auteur] = []
                    if annee not in log[is_part_of][auteur]:
                        log[is_part_of][auteur].append(annee)

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

print(f"\n✓ Terminé — {OUTPUT}")
for revue, auteurs in log.items():
    print(f"  {revue:35s} : {len(auteurs)} poètes de LA trouvés")
    for auteur, annees in list(auteurs.items())[:3]:
        print(f"    {auteur:30s} {sorted(annees)}")

622 poètes chargés


Lecture: 100%|█████████████████████████████| 1.29G/1.29G [04:55<00:00, 4.38Mo/s]


✓ Terminé — jstor_auteurs_revues.json
  poetry                              : 91 poètes de LA trouvés
    Christopher Buckley            [1979, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1994]
    David St. John                 [1977, 1978, 1979, 1981, 1983, 1984, 1986, 1987, 1995, 2001, 2002, 2003]
    Douglas Kearney                [2019, 2020]
  the american poetry review          : 68 poètes de LA trouvés
    Martha Ronk                    [2004]
    Stephanie Brown                [1999, 2001, 2003, 2005, 2011]
    David Trinidad                 [2000, 2006, 2011]
  the kenyon review                   : 33 poètes de LA trouvés
    Marsha de la O                 [2018]
    David St. John                 [1997, 1999, 2009, 2012, 2017, 2020, 2021]
    Carol Muske-Dukes              [2001, 2003, 2009, 2015, 2020]
  prairie schooner                    : 65 poètes de LA trouvés
    Cathy Colman                   [2005]
    David Hernandez                [2002, 2005, 2007]
    Richard Gar

In [2]:
%run croisement_revues_nationales.py

✓ Correction appliquée : Xochitl-Julisa Bermejo — the american poetry review [2015]

622 poètes chargés depuis poetes_LA.csv

✓ CSV écrit : revues_nationales.csv  (622 lignes)

── Présences dans les revues nationales ──
  poetry_magazine                : 67 poètes
  american_poetry_review         : 50 poètes
  kenyon_review                  : 33 poètes
  prairie_schooner               : 56 poètes

── Top 10 poètes (présences cumulées) ──
      nom_canonique  nb_revues                                                          poetry_magazine                     american_poetry_review                              kenyon_review                           prairie_schooner
         Tara Betts          4                                                                   [2015]                                     [2017]                                     [2020]                                     [2016]
Christopher Buckley          4                   [1979, 1986, 1987, 1988, 1989, 1990, 1991, 

In [3]:
# Poètes dans JSTOR Poetry mais pas dans le CSV
import json, pandas as pd

with open("jstor_auteurs_revues.json") as f:
    jstor = json.load(f)

df = pd.read_csv("poetes_LA.csv", encoding="utf-8-sig")
poetes_lower = set(df["nom_canonique"].str.lower().str.strip())

manquants = [nom for nom in jstor["poetry"] if nom.lower() not in poetes_lower]
print(f"{len(manquants)} poètes JSTOR non matchés :")
for nom in manquants:
    print(f"  {nom}")

0 poètes JSTOR non matchés :


In [3]:
#créer le csv
import pandas as pd

df_la = pd.read_csv("poetes_LA.csv", encoding="utf-8-sig")
df_revues = pd.read_csv("revues_nationales.csv", encoding="utf-8-sig")

df_merged = pd.merge(df_la, df_revues.drop(columns=["nom_canonique"]), on="poet_id", how="left")
df_merged.to_csv("poetes_LA_avec_revues.csv", index=False, encoding="utf-8-sig")

print(f"✓ Fichier fusionné : poetes_LA_avec_revues.csv ({len(df_merged)} lignes, {len(df_merged.columns)} colonnes)")
print(df_merged[["nom_canonique", "poetry_magazine", "american_poetry_review", "kenyon_review", "prairie_schooner"]].head(10))

✓ Fichier fusionné : poetes_LA_avec_revues.csv (625 lignes, 25 colonnes)
          nom_canonique poetry_magazine american_poetry_review kenyon_review  \
0            Steve Abee              []                     []            []   
1     Harold Abramowitz              []                     []            []   
2    Ricardo Lira Acuna              []                     []            []   
3          Thomas Adams              []                     []            []   
4         Mikael Ahadou              []                     []            []   
5  Christine Choi Ahmed              []                     []            []   
6         Tanzila Ahmed              []                     []            []   
7         Josephine Ain              []                     []            []   
8       Riua Akinshegun              []                     []            []   
9       Linda Albertano              []                     []            []   

  prairie_schooner  
0               []  
1   

In [4]:
import pandas as pd
df_check = pd.read_csv("poetes_LA.csv", encoding="utf-8-sig")
print(len(df_check))
print(df_check[df_check['nom_canonique'].isin(['Philip Levine', 'Billy Collins', 'Viola Weinberg'])])

622
Empty DataFrame
Columns: [poet_id, nom_canonique, nom_famille, nom_variantes, holdouts, anth_Buk_Vang_1972, streets_inside_1978, poetry_loves_poetry_1985, invocation_LA_1989, grand_passion_1995, stand_up_1990, stand_up_2002, leimert_park_2006, wide_awake_2015, cross_strokes_2015, coiled_serpent_2016, leimert_park_redux_2017, workshop_beyond_baroque_2018, lapl_list, laureat, mohr_blog_absents]
Index: []

[0 rows x 21 columns]


In [5]:
df_revues = pd.read_csv("revues_nationales.csv", encoding="utf-8-sig")
print(len(df_revues))

622


In [6]:
#correction de la cellule 3
import pandas as pd

df_la = pd.read_csv("poetes_LA.csv", encoding="utf-8-sig")
df_revues = pd.read_csv("revues_nationales.csv", encoding="utf-8-sig")

print(f"df_la : {len(df_la)} lignes")
print(f"df_revues : {len(df_revues)} lignes")

df_merged = pd.merge(df_la, df_revues.drop(columns=["nom_canonique"]), on="poet_id", how="left")
print(f"df_merged : {len(df_merged)} lignes")

df_merged.to_csv("poetes_LA_avec_revues.csv", index=False, encoding="utf-8-sig")
print("✓ Sauvegardé")

df_la : 622 lignes
df_revues : 622 lignes
df_merged : 622 lignes
✓ Sauvegardé
